# Baseline Campaign - Node Classification

Datasets: ACM, DBLP, IMDB.

HGB's transductive 20/20/60 split, reported as Macro-F1 and Micro-F1. Five seeds per cell. Every model sees the identical split: the three vendors disagree on how to carve out validation (HGB 20% random, PSHGCN 20% random, HAN 30% sequential) and all three are overridden centrally.

IMDB is multi-label. The campaign binarises with `threshold` (logits > 0), which is what HGB's own runners do; `oracle_topk` reproduces PSHGCN's and H2SGNN's convention and is available for cross-checking published numbers.

Cells 3A and 3B take explicit model lists, so a single cell can be run in isolation. Everything is driven by `scripts/run_baseline_published.py` and `scripts/run_baseline_tuning.py`; nothing is reimplemented here.

## 0. Environment

In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/H2S-NN_test')
else:
    PROJECT_DIR = Path.cwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Working directory:', PROJECT_DIR)


Install the pinned stack. On a fresh Colab runtime this replaces Colab's own torch, so the cell kills the runtime afterwards - **rerun from the first cell once it restarts**. On a machine that already has the pinned versions it is a no-op and execution continues.

In [ ]:
import importlib.metadata
import os
import signal
import subprocess
import sys

requirements_path = PROJECT_DIR / 'requirements.txt'
if not requirements_path.exists():
    raise FileNotFoundError(f'Missing pinned requirements: {requirements_path}')

try:
    installed_torch = importlib.metadata.version('torch')
except importlib.metadata.PackageNotFoundError:
    installed_torch = ''

# requirements.txt carries its own --extra-index-url and -f lines, so this single command
# pulls the CUDA 12.4 torch build and the matching torch_scatter/torch_sparse wheels.
target_ready = installed_torch.startswith('2.6.') and '+cu124' in installed_torch
print('installed torch distribution:', installed_torch or 'not installed')
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--upgrade',
    '-r',
    str(requirements_path),
])

if not target_ready:
    # Colab ships its own torch. Replacing it mid-session leaves the previously imported
    # compiled extensions loaded, so the runtime has to restart before anything else runs.
    print('Pinned Torch/CUDA stack installed. Restarting the runtime; rerun from the first cell.')
    if IN_COLAB:
        os.kill(os.getpid(), signal.SIGKILL)
    raise RuntimeError('Restart Python, then rerun the notebook from the first cell.')


In [ ]:
import torch
import torch_geometric
import torch_scatter
import torch_sparse

if not torch.__version__.startswith('2.6.'):
    raise RuntimeError(f'Expected torch 2.6.*, found {torch.__version__}. Rerun the install cell.')
if torch.version.cuda != '12.4':
    raise RuntimeError(f'Expected the CUDA 12.4 Torch build, found CUDA {torch.version.cuda}.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__)
print('torch_geometric:', torch_geometric.__version__)
print('device:', DEVICE)
if DEVICE == 'cuda':
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', properties.name, '|', round(properties.total_memory / 1024**3, 1), 'GiB')


In [ ]:
# DGL backs HGB's GCN/GAT/HAN, SeHGNN and HINormer. Import the compatibility shim before
# any `import dgl`: some wheels ship no GraphBolt binary for this torch build, and a bare
# import then raises FileNotFoundError on graphbolt_pytorch_2.6.0.dll. The shim detects a
# healthy install and does nothing there.
from src.baselines.runtime import _dgl_compat  # noqa: F401  (import for side effect)
from src.baselines.runtime._dgl_compat import dgl_supports_cuda

import dgl

print('dgl:', dgl.__version__, '| CUDA support:', dgl_supports_cuda())
if not dgl_supports_cuda():
    print('  DGL-backed baselines will run on CPU even though torch has a GPU.')


In [ ]:
# The vendored baselines are cloned under vendor/, not pip-installed.
# This reports which are present before a campaign spends hours discovering a
# missing one. DGL's GraphBolt shim is applied on import where it is needed.
from src.baselines.runtime import available_vendors

missing = [name for name, present in available_vendors().items() if not present]
for name, present in sorted(available_vendors().items()):
    print(('OK  ' if present else 'MISSING '), name)
if missing:
    raise RuntimeError(f'Clone these under vendor/ first: {missing}')



In [ ]:
import wandb

wandb_key = None
if IN_COLAB:
    try:
        from google.colab import userdata
        wandb_key = userdata.get('WANDB_API_KEY')
    except Exception:
        wandb_key = None

if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
else:
    wandb.login()


## 1. Campaign settings

In [ ]:
TASK = 'node_classification'

MODELS = ['all']      # or e.g. ['pshgcn', 'h2sgnn']
DATASETS = ['all']    # or a subset

N_TRIALS = 30
NUM_RUNS = 5
BASE_SEED = 42

# Units are 5 seeds for this task. Partition a campaign across
# machines with these; never run the same index twice concurrently.
UNIT_START = 0
UNIT_COUNT = None

RESUME = False
OUTPUT_DIR = Path('experiments/baselines')
WANDB_PROJECT = 'h2snn_baseline'
WANDB_MODE = 'online'
WANDB_TAGS = ['baselines', TASK]


## 2. Cell matrix

Every (model, dataset) cell, marked `published` (inherits the authors' own settings), `tune` (needs a sweep), or the reason it is not runnable at all.

In [ ]:
from src.baselines.registry import DATASETS as TASK_DATASETS
from src.baselines.registry import (
    get_adapter,
    models_for_task,
    unsupported_reason,
)

rows = []
for spec in models_for_task(TASK):
    for dataset in TASK_DATASETS[TASK]:
        reason = unsupported_reason(spec.name, TASK, dataset)
        if reason:
            status = 'not runnable'
        elif get_adapter(spec.name).published_params(TASK, dataset) is not None:
            status = 'published'
        else:
            status = 'tune'
        rows.append((spec.display, dataset, status, reason or ''))

print(f"{'model':<16}{'dataset':<12}{'status':<14}reason")
for display, dataset, status, reason in rows:
    print(f'{display:<16}{dataset:<12}{status:<14}{reason[:70]}')

from collections import Counter
print()
print(Counter(status for _, _, status, _ in rows))


## 3A. Published-hyperparameter runs

No sweep: these cells inherit settings their authors published for that exact dataset. `--list` first to see which qualify.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, 'scripts/run_baseline_published.py', '--task', TASK, '--list'],
    check=True,
)


In [ ]:
cmd = [
    sys.executable, 'scripts/run_baseline_published.py',
    '--task', TASK,
    '--models', *MODELS,
    '--datasets', *DATASETS,
    '--output_dir', str(OUTPUT_DIR),
    '--num_runs', str(NUM_RUNS),
    '--base_seed', str(BASE_SEED),
    '--unit_start', str(UNIT_START),
]
if UNIT_COUNT is not None:
    cmd += ['--unit_count', str(UNIT_COUNT)]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


## 3B. Tuned runs

Bayes + Hyperband, one sweep per unit, one fresh worker process per trial - the same engine the H2SNN campaign uses. **Validate first**: the printed fit count is the whole cost of the launch.

In [ ]:
tune_cmd = [
    sys.executable, 'scripts/run_baseline_tuning.py',
    '--task', TASK,
    '--models', *MODELS,
    '--datasets', *DATASETS,
    '--output_dir', str(OUTPUT_DIR),
    '--n_trials', str(N_TRIALS),
    '--num_runs', str(NUM_RUNS),
    '--base_seed', str(BASE_SEED),
    '--unit_start', str(UNIT_START),
    '--wandb_project', WANDB_PROJECT,
    '--wandb_mode', WANDB_MODE,
    '--wandb_tags', *WANDB_TAGS,
]
if UNIT_COUNT is not None:
    tune_cmd += ['--unit_count', str(UNIT_COUNT)]
if RESUME:
    tune_cmd.append('--resume')

print('Validating:', ' '.join(tune_cmd + ['--validate_only']))
subprocess.run(tune_cmd + ['--validate_only'], check=True)


In [ ]:
# Launch. Re-run this cell with RESUME = True after a Colab restart: compatible
# per-unit states resume their original sweeps, and incompatible ones are
# preserved while numbered continuation sweeps are created.
print('Starting:', ' '.join(tune_cmd))
subprocess.run(tune_cmd, check=True)


## 4. Aggregate

Collects every finished cell into one CSV plus a LaTeX fragment in the manuscript's column order.

In [ ]:
import json

import pandas as pd

from src.utils.run_reports import report_metrics_for_task

metrics = report_metrics_for_task(TASK)
records = []
for summary_path in sorted(OUTPUT_DIR.glob('*/*_summary.json')):
    payload = json.loads(summary_path.read_text(encoding='utf-8'))
    summary = payload.get('summary', {})
    if summary.get('task') != TASK:
        continue
    model, _, dataset = summary_path.parent.name.partition(f'_{TASK}_')
    row = {
        'model': model,
        'dataset': dataset,
        'source': summary_path.stem.replace('_summary', ''),
        'units': summary.get('num_units', 0),
    }
    for metric in metrics:
        entry = summary.get('metrics', {}).get(metric, {})
        row[metric] = entry.get('mean')
        row[f'{metric}_std'] = entry.get('std')
        if 'std_over_repetitions' in entry:
            row[f'{metric}_std_reps'] = entry['std_over_repetitions']
            row[f'{metric}_std_runs'] = entry['std_over_runs']
    records.append(row)

frame = pd.DataFrame(records)
results_path = Path('results') / f'baselines_{TASK}.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)
frame.to_csv(results_path, index=False)
print('wrote', results_path)
frame


In [ ]:
# LaTeX fragment. Values come straight from the summaries; nothing is filled in
# for a cell that has not run, so a gap stays visibly a gap.
if not frame.empty:
    headline = metrics[0]
    pivot = frame.pivot_table(index='model', columns='dataset', values=headline)
    print(pivot.to_latex(float_format='%.2f', na_rep='--'))
else:
    print('No summaries yet - run cell 3A or 3B first.')


## 5. Resume and partition

Per-unit state lives under `OUTPUT_DIR/<cell>/tuning_state/`. To split one campaign across machines, keep `NUM_RUNS` and `BASE_SEED` fixed and give each machine a disjoint `UNIT_START`/`UNIT_COUNT` range.

In [ ]:
state_root = OUTPUT_DIR / 'tuning_state'
for state_path in sorted(state_root.glob('*/tuning_state_*.json')):
    state = json.loads(state_path.read_text(encoding='utf-8'))
    print(
        f"{state_path.parent.name:<40}{state_path.stem:<28}"
        f"{len(state.get('completed_records', []))}/{state.get('target_trials')} "
        f"sweep {state.get('sweep_id')}"
    )
